# Alpine Rivers Interactive Explorer

This notebook uses **matplotlib controls inside the figure** (text boxes, slider, button).
Click **Download & Refresh** in the figure to load/update data.

In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
from time import perf_counter
import hashlib
import json

import matplotlib.pyplot as plt
from matplotlib.widgets import Button, Slider, TextBox
import numpy as np
import openeo
import xarray as xr

%matplotlib widget

def find_project_root(start: Path) -> Path:
    p = start.resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    return p

ROOT = find_project_root(Path.cwd())
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

def log(msg: str) -> None:
    print(msg)

def validate_inputs(west: float, east: float, south: float, north: float, start: str, end: str) -> None:
    if not (west < east and south < north):
        raise ValueError("Invalid bbox: require west < east and south < north.")
    start_dt = datetime.strptime(start, "%Y-%m-%d")
    end_dt = datetime.strptime(end, "%Y-%m-%d")
    if start_dt > end_dt:
        raise ValueError("Invalid dates: start must be <= end.")

def cache_path_for_query(bbox: dict, start: str, end: str, cloud: float) -> Path:
    payload = {"bbox": bbox, "start": start, "end": end, "cloud": cloud}
    key = hashlib.md5(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:12]
    return DATA_DIR / f"alps_s2_{key}.nc"

def download_cube_if_needed(bbox: dict, start: str, end: str, cloud: float, force: bool) -> Path:
    nc_path = cache_path_for_query(bbox, start, end, cloud)
    if nc_path.exists() and not force:
        log(f"Using cache: {nc_path}")
        return nc_path

    log("Connecting to openEO backend...")
    conn = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

    log("Loading SENTINEL2_L2A (B03, B08)...")
    cube = conn.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=[start, end],
        bands=["B03", "B08"],
        max_cloud_cover=cloud,
    )

    log(f"Downloading to {nc_path} ...")
    cube.download(str(nc_path))
    log("Download done.")
    return nc_path

def compute_ndwi(nc_path: Path):
    ds = xr.open_dataset(nc_path)
    if "B03" not in ds.variables or "B08" not in ds.variables:
        raise RuntimeError("Missing B03/B08 variables in NetCDF.")

    green = ds["B03"].astype("float32")
    nir = ds["B08"].astype("float32")
    ndwi = (green - nir) / (green + nir)

    if "t" in ndwi.dims:
        ndwi2d = ndwi.median(dim="t", skipna=True)
    else:
        ndwi2d = ndwi

    arr = ndwi2d.values
    if not np.isfinite(arr).any():
        raise RuntimeError("No finite NDWI pixels. Try another AOI/date range.")

    lons = ndwi2d["x"].values.astype(float)
    lats = ndwi2d["y"].values.astype(float)
    return arr, lons, lats

def build_mask(arr: np.ndarray, threshold: float) -> np.ndarray:
    mask = np.zeros_like(arr, dtype=np.uint8)
    finite = np.isfinite(arr)
    mask[finite & (arr > threshold)] = 1
    return mask

state = {"arr": None, "lons": None, "lats": None, "ndwi_cb": None}

fig = plt.figure(figsize=(16, 8))
ax_ndwi = fig.add_axes([0.24, 0.55, 0.24, 0.35])
ax_mask = fig.add_axes([0.52, 0.55, 0.24, 0.35])
ax_hist = fig.add_axes([0.80, 0.55, 0.18, 0.35])

status_text = fig.text(0.24, 0.48, "Status: idle", fontsize=10)

def set_status(msg: str) -> None:
    status_text.set_text(f"Status: {msg}")
    fig.canvas.draw_idle()

# Text controls (left panel)
ax_west = fig.add_axes([0.03, 0.88, 0.16, 0.05])
ax_east = fig.add_axes([0.03, 0.82, 0.16, 0.05])
ax_south = fig.add_axes([0.03, 0.76, 0.16, 0.05])
ax_north = fig.add_axes([0.03, 0.70, 0.16, 0.05])
ax_start = fig.add_axes([0.03, 0.62, 0.16, 0.05])
ax_end = fig.add_axes([0.03, 0.56, 0.16, 0.05])
ax_cloud = fig.add_axes([0.03, 0.50, 0.16, 0.05])

tb_west = TextBox(ax_west, "west", initial="11.293602")
tb_east = TextBox(ax_east, "east", initial="11.382866")
tb_south = TextBox(ax_south, "south", initial="46.460163")
tb_north = TextBox(ax_north, "north", initial="46.514768")
tb_start = TextBox(ax_start, "start", initial="2023-07-01")
tb_end = TextBox(ax_end, "end", initial="2023-07-20")
tb_cloud = TextBox(ax_cloud, "cloud%", initial="20")

ax_force = fig.add_axes([0.03, 0.42, 0.16, 0.05])
btn_force = Button(ax_force, "Force: OFF")
force_download = {"value": False}

def on_force(_):
    force_download["value"] = not force_download["value"]
    btn_force.label.set_text("Force: ON" if force_download["value"] else "Force: OFF")
    fig.canvas.draw_idle()

btn_force.on_clicked(on_force)

ax_refresh = fig.add_axes([0.03, 0.34, 0.16, 0.06])
btn_refresh = Button(ax_refresh, "Download & Refresh")

ax_thr = fig.add_axes([0.24, 0.40, 0.52, 0.04])
thr_slider = Slider(ax_thr, "NDWI threshold", -0.2, 0.6, valinit=0.10, valstep=0.01)

fig.text(0.03, 0.30, "Tip: change AOI/time and click Download & Refresh", fontsize=9)

def redraw(threshold: float) -> None:
    arr = state["arr"]
    lons = state["lons"]
    lats = state["lats"]
    if arr is None:
        return

    mask = build_mask(arr, threshold)
    origin = "upper" if lats[0] > lats[-1] else "lower"
    extent = [float(lons.min()), float(lons.max()), float(lats.min()), float(lats.max())]

    ax_ndwi.clear()
    im = ax_ndwi.imshow(arr, cmap="RdBu", vmin=-1, vmax=1, origin=origin, extent=extent)
    ax_ndwi.set_title("NDWI")
    ax_ndwi.set_xlabel("Longitude")
    ax_ndwi.set_ylabel("Latitude")

    if state["ndwi_cb"] is not None:
        state["ndwi_cb"].remove()
    state["ndwi_cb"] = fig.colorbar(im, ax=ax_ndwi, shrink=0.85)

    ax_mask.clear()
    ax_mask.imshow(mask, cmap="Blues", vmin=0, vmax=1, origin=origin, extent=extent)
    ax_mask.set_title(f"Water mask (NDWI > {threshold:.2f})")
    ax_mask.set_xlabel("Longitude")
    ax_mask.set_ylabel("Latitude")

    ax_hist.clear()
    vals = arr[np.isfinite(arr)]
    ax_hist.hist(vals, bins=80, color="gray", alpha=0.85)
    ax_hist.axvline(threshold, color="dodgerblue", linestyle="--", linewidth=2)
    ax_hist.set_title("NDWI histogram")
    ax_hist.set_xlabel("NDWI")
    ax_hist.set_ylabel("Pixel count")

    total = int(np.isfinite(arr).sum())
    water = int(mask.sum())
    pct = 100.0 * water / total if total else 0.0
    set_status(f"loaded | threshold={threshold:.2f} | water={water}/{total} ({pct:.2f}%)")

def on_threshold(val):
    if state["arr"] is not None:
        redraw(float(val))

thr_slider.on_changed(on_threshold)

def on_refresh(_):
    t0 = perf_counter()
    try:
        set_status("validating input...")
        west = float(tb_west.text)
        east = float(tb_east.text)
        south = float(tb_south.text)
        north = float(tb_north.text)
        start = tb_start.text.strip()
        end = tb_end.text.strip()
        cloud = float(tb_cloud.text)

        validate_inputs(west, east, south, north, start, end)
        bbox = {"west": west, "east": east, "south": south, "north": north, "crs": "EPSG:4326"}

        log(f"[{perf_counter()-t0:6.2f}s] Query bbox={bbox}, dates={start}..{end}, cloud<={cloud}%")
        set_status("downloading/opening data...")
        nc_path = download_cube_if_needed(bbox, start, end, cloud, force_download["value"])

        set_status("computing NDWI...")
        arr, lons, lats = compute_ndwi(nc_path)
        state["arr"], state["lons"], state["lats"] = arr, lons, lats

        log(f"[{perf_counter()-t0:6.2f}s] NDWI stats min={np.nanmin(arr):.3f}, max={np.nanmax(arr):.3f}, mean={np.nanmean(arr):.3f}")
        redraw(float(thr_slider.val))
        log(f"[{perf_counter()-t0:6.2f}s] Refresh done.")
    except Exception as exc:
        set_status(f"error - {exc}")
        log(f"ERROR: {exc}")

btn_refresh.on_clicked(on_refresh)

# Auto-load once so plots are not blank
on_refresh(None)
print("Notebook ready. Use figure controls and click 'Download & Refresh'.")